# Qualidade dos Dados — ANP

Neste notebook analiso a qualidade dos dados de preços de combustíveis disponibilizados pela ANP.

O objetivo é investigar e validar identificadores importantes, como CNPJ e CEP, preservando os valores recebidos da fonte e aplicando regras de normalização somente quando a correção puder ser realizada com segurança.

As regras confirmadas durante a análise são implementadas em módulos reutilizáveis dentro de `src/qualidade`, permitindo que posteriormente sejam utilizadas pelo pipeline de dados.

In [2]:
from pathlib import Path 
import sys

import pandas as pd


In [3]:
#Identifico onde o notebook está sendo executado
caminho_atual = Path.cwd()

#Ajusto a raiz caso o Kernel tenha iniciado dentro da pasta notebooks
if caminho_atual.name == "notebooks":
    caminho_projeto = caminho_atual.parent
else:
    caminho_projeto = caminho_atual

print("Diretório atual:", caminho_atual)
print("Raiz do projeto:", caminho_projeto)    

Diretório atual: c:\Users\DataCore\Desktop\projetos_Git\engenharia-dados-combustiveis\notebooks
Raiz do projeto: c:\Users\DataCore\Desktop\projetos_Git\engenharia-dados-combustiveis


In [4]:
#Adiciono a raiz do projeto ao caminho de importação para reutilizar os módulos desenvolvidos em src
if str(caminho_projeto) not in sys.path:
    sys.path.insert(0, str(caminho_projeto))

In [ ]:
#Importar regras de Data Quality

from src.qualidade.validar_cnpj import(
    limpar_cnpj,
    normalizar_cnpj,
    validar_cnpj,

)
from src.qualidade.validar_cep import(
    classificar_cep,
    normalizar_cep,
    validar_formato_cep,
)

In [6]:
#Defino a amostra da ANP utilizada nesta investigação
caminho_arquivo = (
    caminho_projeto
    /"dados"
    /"amostras"
    /"revendas_lpc_2026-09-06_2026-09-12.xlsx"
    )

nome_planilha = "POSTOS REVENDEDORES"
linha_cabecalho = 9

print("Arquivo encontrado:", caminho_arquivo.exists())

Arquivo encontrado: True


In [7]:
# Preservo CNPJ e CEP como texto porque são identificadores,
# e não valores numéricos utilizados em cálculos
tipos_identificadores = {
    "CNPJ": "string",
    "CEP": "string",
}

dados = pd.read_excel(
    caminho_arquivo,
    sheet_name=nome_planilha,
    header=linha_cabecalho,
    dtype=tipos_identificadores,
)

print(f"Quantidade de registros: {len(dados):,}")
print(f"Quantidade de colunas: {len(dados.columns)}")

Quantidade de registros: 19,516
Quantidade de colunas: 15


In [8]:
dados.head()

,CNPJ,RAZÃO,FANTASIA,ENDEREÇO,NÚMERO,COMPLEMENTO,BAIRRO,CEP,MUNICÍPIO,ESTADO,BANDEIRA,PRODUTO,UNIDADE DE MEDIDA,PREÇO DE REVENDA,DATA DA COLETA
0,61602199002409,COMPANHIA ULTRAGAZ S A,ULTRAGAZ,RUA AMARO CASTRO LIMA,1852,NaN,VILA NOVA CAMPO GRANDE,79106361,CAMPO GRANDE,MATO GROSSO DO SUL,ULTRAGAZ,GLP,R$ / 13 kg,100.00,2026-09-07
1,19791896007024,SUPERGASBRAS ENERGIA LTDA,SUPERGASBRAS ENERGIA,AVENIDA DEP. PLINIO RIBEIRO,3220,NaN,JARDIM PALMEIRAS,39402194,MONTES CLAROS,MINAS GERAIS,SUPERGASBRAS ENERGIA,GLP,R$ / 13 kg,120.00,2026-09-09
2,198451000185,COMERCIO DE PETROLEO COLORADO LTDA,TRR COLORADO,RUA DOMINGOS CALAZANS,518,NaN,TRIANGULO,63041130,JUAZEIRO DO NORTE,CEARA,NACIONAL GÁS BUTANO,GLP,R$ / 13 kg,120.00,2026-09-08
3,8220930000162,CODEPE COMBUSTIVEIS LTDA,CODEPE,RODOVIA BR-364 (RIO BRANCO-CRUZEIRO DO SUL) - ...,2755,LOTE 01/02/03/04/05 E 33,PORTAL DA AMAZONIA,69915630,RIO BRANCO,ACRE,VIBRA,GASOLINA COMUM,R$ / litro,7.19,2026-09-08
4,8220930000162,CODEPE COMBUSTIVEIS LTDA,CODEPE,RODOVIA BR-364 (RIO BRANCO-CRUZEIRO DO SUL) - ...,2755,LOTE 01/02/03/04/05 E 33,PORTAL DA AMAZONIA,69915630,RIO BRANCO,ACRE,VIBRA,GASOLINA ADITIVADA,R$ / litro,7.29,2026-09-08


## Qualidade do CNPJ

Nesta etapa preservo o CNPJ recebido da fonte, aplico a normalização necessária para recuperar possíveis zeros à esquerda e valido os dígitos verificadores.

In [9]:
# Preservo o valor original e acompanho cada etapa da transformação
dados["cnpj_original"] = dados["CNPJ"]

dados["cnpj_limpo"] = (
    dados["CNPJ"]
    .apply(limpar_cnpj)
)

dados["cnpj_normalizado"] = (
    dados["CNPJ"]
    .apply(normalizar_cnpj)
)

dados["cnpj_valido"] = (
    dados["cnpj_normalizado"]
    .apply(validar_cnpj)
)

In [10]:
dados[
    [
        "cnpj_original",
        "cnpj_limpo",
        "cnpj_normalizado",
        "cnpj_valido",
    ]
].head(20)

,cnpj_original,cnpj_limpo,cnpj_normalizado,cnpj_valido
0,61602199002409,61602199002409,61602199002409,True
1,19791896007024,19791896007024,19791896007024,True
2,198451000185,198451000185,00198451000185,True
3,8220930000162,8220930000162,08220930000162,True
4,8220930000162,8220930000162,08220930000162,True
5,8220930000162,8220930000162,08220930000162,True
6,8220930000162,8220930000162,08220930000162,True
7,8220930000162,8220930000162,08220930000162,True
8,6003183000185,6003183000185,06003183000185,True
9,6003183000185,6003183000185,06003183000185,True


In [11]:
quantidade_cnpj_valido = dados["cnpj_valido"].sum()
quantidade_cnpj_invalido = len(dados) - quantidade_cnpj_valido

print(f"CNPJs válidos: {quantidade_cnpj_valido:,}")
print(f"CNPJs inválidos: {quantidade_cnpj_invalido:,}")
print(
    f"Percentual válido: "
    f"{quantidade_cnpj_valido / len(dados):.2%}"
)
print(
    f"Percentual inválido: "
    f"{quantidade_cnpj_invalido / len(dados):.2%}"
)

CNPJs válidos: 19,516
CNPJs inválidos: 0
Percentual válido: 100.00%
Percentual inválido: 0.00%


## Qualidade do CEP

Nesta etapa classifico os CEPs conforme a situação recebida da fonte.

CEPs com oito posições são mantidos, CEPs com sete posições podem ser normalizados com zero à esquerda e valores suspeitos não são corrigidos automaticamente.

In [12]:
# Preservo o CEP recebido originalmente da fonte
dados["cep_original"] = dados["CEP"]

# Classifico cada registro antes da transformação
dados["status_cep"] = (
    dados["CEP"]
    .apply(classificar_cep)
)

# Normalizo somente os casos cuja correção é considerada segura
dados["cep_normalizado"] = (
    dados["CEP"]
    .apply(normalizar_cep)
)

# Registro se o CEP pode seguir após a validação estrutural
dados["cep_formato_valido"] = (
    dados["CEP"]
    .apply(validar_formato_cep)
)

In [13]:
dados[
    [
        "cep_original",
        "status_cep",
        "cep_normalizado",
        "cep_formato_valido",
        "MUNICÍPIO",
        "ESTADO",
    ]
].head(20)

,cep_original,status_cep,cep_normalizado,cep_formato_valido,MUNICÍPIO,ESTADO
0,79106361,FORMATO_VALIDO,79106361,True,CAMPO GRANDE,MATO GROSSO DO SUL
1,39402194,FORMATO_VALIDO,39402194,True,MONTES CLAROS,MINAS GERAIS
2,63041130,FORMATO_VALIDO,63041130,True,JUAZEIRO DO NORTE,CEARA
3,69915630,FORMATO_VALIDO,69915630,True,RIO BRANCO,ACRE
4,69915630,FORMATO_VALIDO,69915630,True,RIO BRANCO,ACRE
5,69915630,FORMATO_VALIDO,69915630,True,RIO BRANCO,ACRE
6,69915630,FORMATO_VALIDO,69915630,True,RIO BRANCO,ACRE
7,69915630,FORMATO_VALIDO,69915630,True,RIO BRANCO,ACRE
8,69906380,FORMATO_VALIDO,69906380,True,RIO BRANCO,ACRE
9,69906380,FORMATO_VALIDO,69906380,True,RIO BRANCO,ACRE


In [14]:
dados["status_cep"].value_counts()

status_cep
FORMATO_VALIDO    17655
NORMALIZAVEL       1860
SUSPEITO              1
Name: count, dtype: int64

In [15]:
dados["cep_formato_valido"].value_counts()

cep_formato_valido
True     19515
False        1
Name: count, dtype: int64

In [16]:
quantidade_cep_valido = dados["cep_formato_valido"].sum()
quantidade_cep_rejeitado = len(dados) - quantidade_cep_valido

print(f"CEPs estruturalmente válidos: {quantidade_cep_valido:,}")
print(f"CEPs rejeitados: {quantidade_cep_rejeitado:,}")
print(
    f"Percentual válido: "
    f"{quantidade_cep_valido / len(dados):.4%}"
)
print(
    f"Percentual rejeitado: "
    f"{quantidade_cep_rejeitado / len(dados):.4%}"
)

CEPs estruturalmente válidos: 19,515
CEPs rejeitados: 1
Percentual válido: 99.9949%
Percentual rejeitado: 0.0051%
